<a href="https://colab.research.google.com/github/Sayed-Musaraf-Ali/video-transcription-ai/blob/main/AI_vedioToText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# safe setup: installs gradio/transformers/ffmpeg, NEVER torch (so a GPU build can't be broken)
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q gradio transformers

import subprocess, torch
has_gpu = (subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0) and torch.cuda.is_available()
print("torch:", torch.__version__)
print("using:", "GPU (fast)" if has_gpu else "CPU (works, a little slower)")
print("ffmpeg:", subprocess.run(["ffmpeg","-version"], capture_output=True, text=True).stdout.splitlines()[0][:30])
print("=> PROCEED to Cell 2 (no menu clicks needed)")

torch: 2.11.0+cu128
using: GPU (fast)
ffmpeg: ffmpeg version 4.4.2-0ubuntu0.
=> PROCEED to Cell 2 (no menu clicks needed)


In [2]:
import os, re, cv2, numpy as np, urllib.request, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

vid = "sample.mp4"
if not os.path.exists(vid):
    urllib.request.urlretrieve("https://www.w3schools.com/html/mov_bbb.mp4", vid)
    print("downloaded", vid)

# EXACT name - image GIT, no broken temporal weights. DO NOT change this string.
processor = AutoProcessor.from_pretrained("microsoft/git-base")
git = AutoModelForCausalLM.from_pretrained("microsoft/git-base").to(device)
print("model loaded on", device)

cap = cv2.VideoCapture(vid)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idx = np.linspace(0, total - 1, 4, dtype=int)   # 4 timestamps across the clip

captions = []
for i in idx:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
    ok, f = cap.read()
    if not ok:
        continue
    arr = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
    if arr.std() < 5:          # skip black / blank frames
        continue
    fr = Image.fromarray(arr)
    inp = processor(images=fr, return_tensors="pt").to(device)
    ids = git.generate(pixel_values=inp.pixel_values, max_length=30,
                       min_length=5, num_beams=3, no_repeat_ngram_size=2)
    txt = processor.batch_decode(ids, skip_special_tokens=True)[0]
    txt = re.sub(r"\[unused\d+\]", "", txt).strip()   # safety net
    if txt and txt not in captions:
        captions.append(txt)
    print("frame caption:", txt)
cap.release()

print("COMBINED:", " | ".join(captions))

downloaded sample.mp4


preprocessor_config.json:   0%|          | 0.00/503 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  707MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/305 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

model loaded on cuda
frame caption: this is a dog that looks like person.
frame caption: animal in the grass
frame caption: film character in a field of green grass
frame caption: film character in a field of grass
COMBINED: this is a dog that looks like person. | animal in the grass | film character in a field of green grass | film character in a field of grass


In [3]:
# 1. install Whisper (the speech-to-text model) - run once
!pip install -q openai-whisper
import os, re, subprocess, tempfile, torch
import gradio as gr
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"
wmodel = whisper.load_model("base", device=device)   # 'base' is fast; change to 'small' for better accuracy
print("Whisper loaded on", device)

def clean_audio(in_path):
    # keep AUDIO only, 16kHz mono wav -> Whisper reads any container after this
    out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    subprocess.run(["ffmpeg", "-y", "-i", in_path, "-vn", "-ac", "1", "-ar", "16000", out],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out

def vpath(x):
    if x is None:
        return None
    return x.name if hasattr(x, "name") else str(x)

def transcribe(video):
    p = vpath(video)
    if p is None or not os.path.exists(p):
        return "(no file)", "(no file)"
    wav = clean_audio(p)
    try:
        result = wmodel.transcribe(wav, fp16=(device == "cuda"))
    except Exception as e:
        return "(transcribe error: " + str(e) + ")", ""
    text = result["text"].strip()
    # build a timestamped subtitle list (great for the report)
    lines = []
    for seg in result.get("segments", []):
        s = int(seg["start"]); e = int(seg["end"])
        lines.append("[{:02d}:{:02d}-{:02d}:{:02d}] {}".format(s // 60, s % 60, e // 60, e % 60, seg["text"].strip()))
    return text, "\n".join(lines)

gr.Interface(
    fn=transcribe,
    inputs=gr.File(label="Upload your video (we will LISTEN to it)"),
    outputs=[
        gr.Textbox(label="What the person SAID (transcript)"),
        gr.Textbox(label="Timestamped subtitles"),
    ],
    title="AI Video to Text - SPEECH transcription (Whisper)",
).launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 60.2MiB/s]


Whisper loaded on cuda
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://96f7e78cec448c0bea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
